# 1 — Small-dictionary identifiability

This report asks whether the intended deconvolution is identifiable before scaling: can the Torch baseline recover sparse mixtures when the selected global-catalogue pool contains 10, 50, or 100 candidate ions?

## Method

`deconvolution.identifiability` deterministically chooses a 100-ion pool from the configured global condition. For each size and repeat it selects a seeded subdictionary, generates exact synthetic abundances, runs projected-gradient recovery, and stores rank, maximum coherence, abundance MAE, reconstruction MSE, and exact-support fraction.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from msi_autoencoder_wrapper.analysis.precompute.cli import run_precompute_command
from msi_autoencoder_wrapper.visualization.metrics import plot_violin_with_points
from msi_autoencoder_wrapper.visualization.theme import resolve_theme

REPOSITORY_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file())
NOTEBOOK_DIR = REPOSITORY_ROOT / 'assets/experiments/autoencoder_architecture/notebooks/annotation_model/19_09_deconvolution_inital'
SETTINGS_PATH = NOTEBOOK_DIR / 'analysis_settings.yaml'
RESULTS_DIR = NOTEBOOK_DIR / 'part_1_identifiability_small_dictionaries_results'
print(run_precompute_command(SETTINGS_PATH, background=False))

## Reproducible execution

Run the printed command once. Exact pool candidate records are retained in `selected_candidates.csv`; all random subset and synthetic-mixture draws are fixed by the YAML seeds.

In [ ]:
results_path = RESULTS_DIR / 'identifiability.csv'
if not results_path.is_file():
    raise FileNotFoundError(f'Missing {results_path}. Run the command printed above.')
identifiability = pd.read_csv(results_path)
identifiability.groupby('candidate_count', as_index=False).agg(
    abundance_mae=('abundance_mae', 'mean'),
    reconstruction_mse=('reconstruction_mse', 'mean'),
    support_exact_fraction=('support_exact_fraction', 'mean'),
    minimum_rank=('dictionary_rank', 'min'),
    maximum_coherence=('max_pairwise_coherence', 'max'),
)

In [ ]:
metric = 'abundance_mae'
candidate_counts = sorted(identifiability['candidate_count'].unique())
theme = resolve_theme(None)
figure, axis = plt.subplots(figsize=theme.figure_size, dpi=theme.figure_dpi)
for position, candidate_count in enumerate(candidate_counts, start=1):
    values = identifiability.loc[identifiability['candidate_count'].eq(candidate_count), metric].to_numpy()
    plot_violin_with_points(values, position=position, ax=axis, color=theme.color_for_model(f'C={candidate_count}', position - 1), label=f'C={candidate_count}')
axis.set_xticks(np.arange(1, len(candidate_counts) + 1), [f'C={count}' for count in candidate_counts])
axis.set(xlabel='Candidate subdictionary size', ylabel='Abundance MAE', title='Recovery across seeded candidate subsets')
axis.legend()
figure.savefig(RESULTS_DIR / 'abundance_mae_distribution.png', bbox_inches='tight')
figure

## Decision criterion

Rank deficiency or coherence near one means that the impulse-rendered dictionary cannot uniquely resolve the relevant candidates; this is a dictionary-design limitation, not evidence for adding learned LISTA parameters. Stable rank, low residuals, and improving recovery at 10→50→100 justify the next scale-up.